# EX02. [추가연습] 다중분류 실전 문제 - 아이리스 품종 분류

> 메인 노트북(07, 10번)에서는 Wine 데이터로 다중분류를 연습했습니다. 여기서는 **다른 데이터셋(Iris)**으로 같은 유형의 문제를 다시 풀어보며 '문제 유형에 대한 응용력'을 기릅니다.

### 사용 데이터
`data/iris.csv` - 꽃잎/꽃받침 길이·너비로 붓꽃 품종 3종(setosa/versicolor/virginica, 0/1/2)을 분류

## 목차
1. 데이터 로딩 & EDA
2. 전처리
3. 머신러닝 다중분류 모델링
4. 딥러닝 다중분류 모델링
5. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

iris = pd.read_csv('../data/iris.csv')
iris.head()

## 1. 데이터 로딩 & EDA

### 📖 이론 설명
클래스 불균형 여부를 가장 먼저 확인합니다. Iris는 각 품종이 정확히 50개씩으로, **균형 잡힌 데이터셋**입니다 - 06/09번에서 다룬 불균형 문제와 대비해서 보면 좋습니다.

### 💻 예제 코드

In [ ]:
print(iris['target'].value_counts())
sns.countplot(x='target', data=iris)
plt.title('Class distribution (balanced)')
plt.show()

sns.pairplot(iris, hue='target', vars=['sepal length (cm)', 'petal length (cm)'])
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `petal length (cm)`과 `petal width (cm)`의 관계를 `target`으로 색을 나눠 scatterplot으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sns.scatterplot(data=iris, x='petal length (cm)', y='petal width (cm)', hue='target')  # hue로 품종별 색을 나누면 두 변수만으로도 품종이 잘 구분되는지 한눈에 보임
plt.show()
```

</details>

## 2. 전처리

### 📖 이론 설명
결측치가 없고 모든 변수가 비슷한 스케일(cm 단위)이라 전처리 부담이 적은 데이터입니다. 그래도 모델링 습관을 위해 스케일링과 분할은 동일하게 진행합니다.

### 💻 예제 코드

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = iris.drop(columns=['target'])
y = iris['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train.shape

## 3. 머신러닝 다중분류 모델링

### 📖 이론 설명
07번 노트북과 동일한 흐름입니다 - 평가지표에 `average` 옵션을 지정하는 것을 잊지 마세요.

### 💻 예제 코드

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, classification_report

logi = LogisticRegression(max_iter=1000)
logi.fit(X_train_scaled, y_train)
logi_pred = logi.predict(X_test_scaled)
print('LogisticRegression f1(macro):', f1_score(y_test, logi_pred, average='macro'))

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
print(classification_report(y_test, rf_pred))

### ✍️ TODO: 실전 문제

**문제 1.** `SVC(kernel='linear', probability=True)`를 학습시켜 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
svc = SVC(kernel='linear', probability=True, random_state=42)  # kernel='linear': 직선(초평면)으로 클래스를 나누는 가장 단순한 형태의 SVM
svc.fit(X_train_scaled, y_train)
print(accuracy_score(y_test, svc.predict(X_test_scaled)))
```

</details>

## 4. 딥러닝 다중분류 모델링

### 📖 이론 설명
10번 노트북과 동일하게, 출력층 노드=클래스 수(3), activation='softmax'로 설계합니다.

### 💻 예제 코드

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(100)
class_num = y.nunique()

model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X_train_scaled.shape[1],)))
model.add(Dense(8, activation='relu'))
model.add(Dense(class_num, activation='softmax'))
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train_scaled, y_train, epochs=100, batch_size=8,
                     validation_data=(X_test_scaled, y_test), callbacks=[es], verbose=0)

pred_label = np.argmax(model.predict(X_test_scaled), axis=1)
from sklearn.metrics import accuracy_score
print('DL accuracy:', accuracy_score(y_test, pred_label))

## 5. 종합 실전 문제

**다중분류 미니 모의고사입니다.**

**문제 1.** `RandomForestClassifier`의 confusion matrix를 heatmap으로 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, rf_pred)  # 클래스가 3개이므로 3x3 행렬이 반환됨
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('Actual')
plt.show()
```

</details>

**문제 2.** 딥러닝 모델의 loss curve를 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
plt.plot(history.history['loss'], label='train')
plt.plot(history.history['val_loss'], label='val')  # 두 곡선이 함께 내려가면 정상적으로 학습되고 있다는 뜻
plt.legend(); plt.title('Loss Curve'); plt.show()
```

</details>